In [ ]:
# Install libraries
!pip install earthengine-api geemap

# Import
import ee
import geemap

# Authenticate & Initialize
ee.Authenticate()
ee.Initialize(project='handy-post-458213-t7')

# -------------------------------
# 📍 Farm Boundary (your polygon)
# -------------------------------
roi = ee.Geometry.Polygon([
    [
        [75.619846, 16.287265],
        [75.61976, 16.287749],
        [75.623441, 16.287816],
        [75.623447, 16.287322],
        [75.619846, 16.287265]
    ]
])

# -------------------------------
# 🛰️ Load Sentinel-2 Image
# -------------------------------
image = (ee.ImageCollection('COPERNICUS/S2_SR')
         .filterBounds(roi)
         .filterDate('2024-01-01', '2024-03-01')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
         .sort('CLOUDY_PIXEL_PERCENTAGE')
         .first())

# -------------------------------
# 🌿 Calculate EVI
# -------------------------------
evi = image.expression(
    '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
    {
        'NIR': image.select('B8'),
        'RED': image.select('B4'),
        'BLUE': image.select('B2')
    }
).rename('EVI')

# -------------------------------
# 🎨 Visualization Settings
# -------------------------------
evi_vis = {
    'min': -0.1,
    'max': 0.6,
    'palette': ['brown', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# -------------------------------
# 🗺️ Create Map
# -------------------------------
Map = geemap.Map(center=[16.2875, 75.6215], zoom=17)

# Add Satellite Layer
Map.addLayer(image, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Satellite')

# Add EVI Layer
Map.addLayer(evi.clip(roi), evi_vis, 'EVI')

# Add Boundary
Map.addLayer(roi, {'color': 'red'}, 'Farm Boundary')

# -------------------------------
# 📊 Add Legend
# -------------------------------
legend_dict = {
    'No Vegetation': 'brown',
    'Low Growth': 'yellow',
    'Moderate': 'lightgreen',
    'Healthy': 'green',
    'Dense Crop': 'darkgreen'
}

Map.add_legend(
    title="EVI Legend",
    keys=['No Vegetation', 'Low Growth', 'Moderate', 'Healthy', 'Dense Crop'],
    colors=[
        (165, 42, 42),     # brown
        (255, 255, 0),     # yellow
        (144, 238, 144),   # lightgreen
        (0, 128, 0),       # green
        (0, 100, 0)        # darkgreen
    ]
)

# -------------------------------
# 📈 Debug (Check EVI value)
# -------------------------------
print("Mean EVI:",
      evi.reduceRegion(
          reducer=ee.Reducer.mean(),
          geometry=roi,
          scale=10
      ).getInfo())

# Show Map
Map

Mean EVI: {'EVI': 0.3894760133978207}


Map(center=[16.2875, 75.6215], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…

In [ ]:
# Install libraries
!pip install earthengine-api geemap

# Import
import ee
import geemap

# Authenticate & Initialize
ee.Authenticate()
ee.Initialize(project='handy-post-458213-t7')

# -------------------------------
# 📍 Farm Boundary
# -------------------------------
roi = ee.Geometry.Polygon([
    [
        [75.619846, 16.287265],
        [75.61976, 16.287749],
        [75.623441, 16.287816],
        [75.623447, 16.287322],
        [75.619846, 16.287265]
    ]
])

# -------------------------------
# 🛰️ Load Sentinel-2 Image
# -------------------------------
image = (ee.ImageCollection('COPERNICUS/S2_SR')
         .filterBounds(roi)
         .filterDate('2024-01-01', '2024-03-01')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
         .sort('CLOUDY_PIXEL_PERCENTAGE')
         .first())

# -------------------------------
# 🌿 Calculate NDVI
# -------------------------------
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# -------------------------------
# 🎨 Visualization Settings
# -------------------------------
ndvi_vis = {
    'min': 0,
    'max': 1,
    'palette': [
        'brown',      # soil
        'yellow',
        'lightgreen',
        'green',
        'darkgreen'
    ]
}

# -------------------------------
# 🗺️ Create Map
# -------------------------------
Map = geemap.Map(center=[16.2875, 75.6215], zoom=17)

# Satellite Layer
Map.addLayer(image, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Satellite')

# NDVI Layer
Map.addLayer(ndvi.clip(roi), ndvi_vis, 'NDVI')

# Boundary
Map.addLayer(roi, {'color': 'red'}, 'Farm Boundary')

# -------------------------------
# 📊 Legend (FIXED VERSION)
# -------------------------------
Map.add_legend(
    title="NDVI Legend",
    keys=['No Vegetation', 'Low', 'Moderate', 'Healthy', 'Dense'],
    colors=[
        (165, 42, 42),     # brown
        (255, 255, 0),     # yellow
        (144, 238, 144),   # lightgreen
        (0, 128, 0),       # green
        (0, 100, 0)        # darkgreen
    ]
)

# -------------------------------
# 📈 Debug Value
# -------------------------------
print("Mean NDVI:",
      ndvi.reduceRegion(
          reducer=ee.Reducer.mean(),
          geometry=roi,
          scale=10
      ).getInfo())

# Show Map
Map

Mean NDVI: {'NDVI': 0.1197498157968703}


Map(center=[16.2875, 75.6215], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…